# ChotuGPT in Google Colab — the whole course in one notebook

Bonus lecture 34 of the **LLM Masterclass** (Programming Line). This notebook builds ChotuGPT from scratch:
**data → pretraining → supervised fine-tuning → chat**.

* Free GPU: *Runtime → Change runtime type → T4 GPU*. On a CPU it works too, just slower.
* `QUICK = True` trains for 300 steps (about a minute on a GPU or CPU). `QUICK = False` is the full
  1,500-step run from lecture 24.
* On a GPU the numbers differ slightly from the video (which ran on a CPU), but the story is the same.

In [ ]:
!git clone https://github.com/programmingline/chotugpt.git
%cd chotugpt
!pip install -q -r requirements.txt

In [ ]:
import os, runpy, time
import torch
from chotugpt.data import load_ids, prepare
from chotugpt.generate import generate
from chotugpt.model import GPT, GPTConfig
from chotugpt.sft import END, QA_PAIRS, finetune, make_example, prompt_text
from chotugpt.train import complete, load_checkpoint, save_checkpoint, set_seed, train

QUICK = True                                   # False = the full 1,500-step run
STEPS = 300 if QUICK else 1500
NAME = "chotugpt_quick" if QUICK else "chotugpt"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| steps:", STEPS)

## 1. Data (lecture 23): make the chai-diary corpus, clean it, split it, train the tokenizer

In [ ]:
runpy.run_path("data/make_corpus.py", run_name="__main__")
r = prepare()
tok = r["tokenizer"]
print(len(r["train_ids"]), "train tokens /", len(r["val_ids"]), "val tokens | vocab", tok.vocab_size)
print(tok.tokens("After my morning chai, I feel fresh."))

## 2. Pretraining (lecture 24)

In [ ]:
set_seed(24)
train_data = torch.from_numpy(load_ids("train").astype("int64"))
val_data = torch.from_numpy(load_ids("val").astype("int64"))
model = GPT(GPTConfig()).to(device)
print(f"{model.num_params():,} parameters")
history = train(model, train_data, val_data, max_steps=STEPS, batch_size=8, lr=1e-3,
                on_eval=lambda s, tr, va, t: print(f"step {s:5d}  train {tr:.4f}  val {va:.4f}  {t:5.0f}s"))
save_checkpoint(model, f"{NAME}.pt")
model.eval()
print("After my morning chai, I feel ->", repr(complete(model, tok, "After my morning chai, I feel", 1)))

## 3. Supervised fine-tuning (lecture 26): from text continuer to a tiny assistant

In [ ]:
set_seed(26)
sft = load_checkpoint(f"{NAME}.pt", device)
for step, loss in finetune(sft, [make_example(tok, q, a) for q, a in QA_PAIRS], steps=150):
    print(f"sft step {step:3d}  loss {loss:.4f}")
save_checkpoint(sft, f"{NAME}_sft.pt")

## 4. Chat (lecture 28's sampling: temperature 0.7, top-k 10)

In [ ]:
def chat(question, seed=0, max_new_tokens=40):
    ids = tok.encode(prompt_text(question))
    g = torch.Generator(device=device).manual_seed(seed)
    out = generate(sft, torch.tensor([ids], device=device), max_new_tokens,
                   temperature=0.7, top_k=10, generator=g)
    return tok.decode(out[0, len(ids):].tolist()).split(END)[0].strip()

for q in ["How do you feel after morning chai?", "What do you put in chai?", "How do you feel after lunch?"]:
    print("you:", q, "\nChotuGPT:", chat(q))

In [ ]:
# Your turn: type a question (an empty line stops). ChotuGPT only knows its tiny chai-diary world!
while True:
    q = input("you: ").strip()
    if not q:
        break
    print("ChotuGPT:", chat(q, seed=int(time.time())))

## The same in one command

```
!python lectures/lecture34_full_run.py --quick
```